<a href="https://colab.research.google.com/github/swathi22101997/capstone-project/blob/main/capstone_project_module_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zepto Data & AI Platform

module-1



1.Using the requests and BeautifulSoup libraries, scrape all books listed across at least 3 different book categories (or, if you prefer, the first 5 paginated listing pages of the "All products" catalogue — either scope is acceptable as long as your final dataset has at least 60 books). For each book capture: title, price (as listed, in GBP), star_rating (as text, e.g. "Three"), availability (as listed text), and category.



In [ ]:
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://books.toscrape.com/catalogue/"
books_data = []

# Scraping the first 3 paginated pages (20 books per page = 60 books total)
for page_num in range(1, 4):
    page_url = f"{BASE_URL}page-{page_num}.html"
    response = requests.get(page_url)

    if response.status_code != 200:
        print(f"Failed to fetch page {page_num}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")
    articles = soup.find_all("article", class_="product_pod")

    for article in articles:
        # 1. Title
        title = article.h3.a["title"]

        # 2. Price (in GBP, as listed)
        price_text = article.find("p", class_="price_color").text.strip()

        # 3. Star Rating (Extract rating class e.g., "Three")
        rating_classes = article.find("p", class_="star-rating")["class"]
        star_rating = [c for c in rating_classes if c != "star-rating"][0]

        # 4. Availability
        availability = (
            article.find("p", class_="instock availability").text.strip()
        )

        # 5. Category (Fetch from book's detail page breadcrumbs)
        book_rel_url = article.h3.a["href"]
        book_detail_url = BASE_URL + book_rel_url
        detail_response = requests.get(book_detail_url)
        detail_soup = BeautifulSoup(detail_response.content, "html.parser")

        # Breadcrumbs format: Home > Books > [Category Name] > [Title]
        breadcrumbs = detail_soup.find("ul", class_="breadcrumb").find_all("a")
        category = (
            breadcrumbs[2].text.strip() if len(breadcrumbs) >= 3 else "Unknown"
        )

        # Store collected fields
        books_data.append(
            {
                "title": title,
                "price": price_text,
                "star_rating": star_rating,
                "availability": availability,
                "category": category,
            }
        )

    print(f"Page {page_num} scraped successfully. Total books: {len(books_data)}")
    time.sleep(0.5)  # Politeness delay

# Displaying sample extracted data
print(f"\nSuccessfully collected {len(books_data)} books.\n")
for b in books_data[:3]:
    print(b)

Page 1 scraped successfully. Total books: 20
Page 2 scraped successfully. Total books: 40
Page 3 scraped successfully. Total books: 60

Successfully collected 60 books.

{'title': 'A Light in the Attic', 'price': '£51.77', 'star_rating': 'Three', 'availability': 'In stock', 'category': 'Poetry'}
{'title': 'Tipping the Velvet', 'price': '£53.74', 'star_rating': 'One', 'availability': 'In stock', 'category': 'Historical Fiction'}
{'title': 'Soumission', 'price': '£50.10', 'star_rating': 'One', 'availability': 'In stock', 'category': 'Fiction'}


2.Clean the scraped fields into proper types:

Strip the currency symbol from price and convert it to a float column price_gbp.
Convert the text star rating (One…Five) into an integer column rating (1–5).
Parse the availability text into a boolean column in_stock.
If any field fails to parse for a given row (e.g., unexpected text), handle it with the median-imputation approach for numeric fields or drop the row (state and justify your choice) — do not leave the pipeline crashing on messy rows.

In [ ]:
import pandas as pd
import numpy as np

def clean_scraped_books(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans raw scraped book data:
    - Strips currency symbol and converts `price` to float (`price_gbp`).
    - Converts text star ratings ('One'...'Five') to integer `rating` (1-5).
    - Parses `availability` text into a boolean `in_stock`.
    - Handles parsing errors using median imputation for numeric fields.
    """
    # Work on a copy to avoid SettingWithCopyWarning
    cleaned_df = df.copy()

    # 1. Clean and convert Price to float (price_gbp)
    # Extract numeric float values (handles currency symbols like £, $, etc.)
    cleaned_df['price_gbp'] = (
        cleaned_df['price']
        .astype(str)
        .str.extract(r'(\d+\.\d+|\d+)', expand=False)
    )
    cleaned_df['price_gbp'] = pd.to_numeric(cleaned_df['price_gbp'], errors='coerce')

    # Impute missing/corrupted price_gbp values with the median price
    median_price = cleaned_df['price_gbp'].median()
    cleaned_df['price_gbp'] = cleaned_df['price_gbp'].fillna(median_price)

    # 2. Convert text Star Rating to Integer (rating)
    rating_map = {
        'one': 1,
        'two': 2,
        'three': 3,
        'four': 4,
        'five': 5
    }

    cleaned_df['rating'] = (
        cleaned_df['star_rating']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(rating_map)
    )

    # Impute missing/corrupted star ratings with the median rating
    median_rating = cleaned_df['rating'].median()
    cleaned_df['rating'] = cleaned_df['rating'].fillna(median_rating).astype(int)

    # 3. Parse Availability text into Boolean (in_stock)
    # Checks if 'in stock' or 'available' is present in the text
    cleaned_df['in_stock'] = (
        cleaned_df['availability']
        .astype(str)
        .str.lower()
        .str.contains('in stock', na=False)
    )

    # Drop original raw columns if desired, keeping clean types
    # cleaned_df = cleaned_df.drop(columns=['price', 'star_rating', 'availability'])

    return cleaned_df

# --- Example Usage ---
if __name__ == "__main__":
    raw_data = {
        'title': ['Book A', 'Book B', 'Book C', 'Book D'],
        'price': ['£51.77', '£10.00', 'invalid_price', '£22.50'],
        'star_rating': ['Three', 'Five', 'One', 'Unknown'],
        'availability': ['In stock', 'In stock (19 available)', 'Out of stock', 'In stock'],
        'category': ['Travel', 'Mystery', 'Travel', 'Sequential Art']
    }

    df_raw = pd.DataFrame(raw_data)
    df_clean = clean_scraped_books(df_raw)
    print(df_clean[['title', 'price_gbp', 'rating', 'in_stock', 'category']])
    print("\nData Types:\n", df_clean[['price_gbp', 'rating', 'in_stock']].dtypes)

    title  price_gbp  rating  in_stock        category
0  Book A      51.77       3      True          Travel
1  Book B      10.00       5      True         Mystery
2  Book C      22.50       1     False          Travel
3  Book D      22.50       3      True  Sequential Art

Data Types:
 price_gbp    float64
rating         int64
in_stock        bool
dtype: object


3.Convert price_gbp to a price_inr column using the project's fixed baseline conversion rate: 1 GBP = 105.50 INR. This is an artificial, project-defined constant for this assignment, not a live or historical market rate, so it never needs a lookup or a date reference. This fixed-rate conversion is the required, keyless baseline and is what gets graded for this task — it requires no external API call and no network access; simply state this exact rate in your README. (Optional, ungraded stretch — must not affect your required submission: if you want extra practice with the requests library and explicit HTTP status-code handling, you may additionally look up any free, keyless currency-conversion API of your own choosing, check its response status code explicitly, and fall back to the fixed rate above on any failure. This is entirely optional; your price_inr column must be fully correct using only the required fixed-rate baseline, since that path alone is what gets graded.)



In [ ]:
import pandas as pd
import requests

# 1. Define the required project baseline conversion rate
FIXED_GBP_TO_INR = 105.50

def convert_gbp_to_inr(df: pd.DataFrame, use_api_stretch: bool = False) -> pd.DataFrame:
    """
    Converts 'price_gbp' to 'price_inr'.

    Required graded baseline: Uses fixed rate (1 GBP = 105.50 INR).
    Optional stretch goal: Attempts fetching live rate via API, with explicit
    HTTP status code handling, falling back to the fixed baseline on failure.
    """
    df = df.copy()
    conversion_rate = FIXED_GBP_TO_INR

    # Optional Stretch Goal: Fetch rate from a free keyless API
    if use_api_stretch:
        api_url = "https://open.er-api.com/v6/latest/GBP"
        try:
            response = requests.get(api_url, timeout=5)
            # Explicitly check HTTP status code
            if response.status_code == 200:
                data = response.json()
                if data.get("result") == "success" and "INR" in data.get("rates", {}):
                    conversion_rate = float(data["rates"]["INR"])
                    print(f"API success (Status 200): Using live rate 1 GBP = {conversion_rate} INR")
                else:
                    print(f"API response missing required data. Falling back to fixed rate: {FIXED_GBP_TO_INR}")
            else:
                print(f"API error (Status Code: {response.status_code}). Falling back to fixed rate: {FIXED_GBP_TO_INR}")
        except requests.RequestException as e:
            print(f"Network request failed ({e}). Falling back to fixed rate: {FIXED_GBP_TO_INR}")

    # Calculate price_inr using the determined conversion rate and round to 2 decimal places
    df['price_inr'] = (df['price_gbp'] * conversion_rate).round(2)

    return df

# --- Example Usage ---
if __name__ == "__main__":
    sample_data = pd.DataFrame({
        'title': ['Book A', 'Book B', 'Book C'],
        'price_gbp': [51.77, 10.00, 22.50]
    })

    # Required graded baseline run
    df_result = convert_gbp_to_inr(sample_data, use_api_stretch=False)
    print(df_result[['title', 'price_gbp', 'price_inr']])

    title  price_gbp  price_inr
0  Book A      51.77    5461.74
1  Book B      10.00    1055.00
2  Book C      22.50    2373.75


4.Design a normalized SQLite schema with at least two tables sharing a primary/foreign key relationship, for example:

categories(category_id INTEGER PRIMARY KEY, category_name TEXT UNIQUE)
books(book_id INTEGER PRIMARY KEY, title TEXT, price_gbp REAL, price_inr REAL, rating INTEGER, in_stock INTEGER, category_id INTEGER REFERENCES categories(category_id))
(You may rename columns/tables, but the two-table PK/FK structure is required.)

In [ ]:
import sqlite3
import pandas as pd

def initialize_database(db_name: str = "books_data.db") -> sqlite3.Connection:
    """
    Connects to SQLite and creates the normalized two-table schema with PK/FK constraints.
    """
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Enable foreign key constraint support in SQLite
    cursor.execute("PRAGMA foreign_keys = ON;")

    # Drop existing tables if re-running script to ensure clean schema creation
    cursor.execute("DROP TABLE IF EXISTS books;")
    cursor.execute("DROP TABLE IF EXISTS categories;")

    # 1. Create Parent Table: categories
    cursor.execute("""
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT NOT NULL UNIQUE
        );
    """)

    # 2. Create Child Table: books
    cursor.execute("""
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
            in_stock INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id) REFERENCES categories (category_id)
                ON DELETE CASCADE
                ON UPDATE CASCADE
        );
    """)

    conn.commit()
    return conn


def load_data_into_schema(cleaned_df: pd.DataFrame, conn: sqlite3.Connection):
    """
    Populates the normalized categories and books tables using the cleaned pandas DataFrame.
    """
    cursor = conn.cursor()

    # Step A: Insert unique categories into parent table
    unique_categories = cleaned_df[['category']].drop_duplicates().dropna()
    for _, row in unique_categories.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO categories (category_name) VALUES (?);",
            (row['category'],)
        )
    conn.commit()

    # Step B: Map category names to their generated category_id
    category_map = pd.read_sql("SELECT category_id, category_name FROM categories;", conn)
    cat_to_id = dict(zip(category_map['category_name'], category_map['category_id']))

    # Step C: Prepare and insert books into child table
    books_data = []
    for _, row in cleaned_df.iterrows():
        cat_id = cat_to_id.get(row['category'])
        books_data.append((
            row['title'],
            float(row['price_gbp']),
            float(row['price_inr']),
            int(row['rating']),
            int(row['in_stock']), # Boolean 0 or 1
            cat_id
        ))

    cursor.executemany("""
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?);
    """, books_data)

    conn.commit()


# --- Example Pipeline Usage ---
if __name__ == "__main__":
    # Sample cleaned data matching Task 1 - Task 3 output
    sample_cleaned_data = pd.DataFrame({
        'title': ['A Light in the Attic', 'Tipping the Velvet', 'Soumission'],
        'price_gbp': [51.77, 53.74, 50.10],
        'price_inr': [5461.73, 5669.57, 5285.55],
        'rating': [3, 1, 1],
        'in_stock': [1, 1, 1],
        'category': ['Poetry', 'Historical Fiction', 'Fiction']
    })

    # Initialize DB and populate
    connection = initialize_database("books_catalog.db")
    load_data_into_schema(sample_cleaned_data, connection)

    # Verify tables created
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)
    print("Tables in Database:\n", tables)

    # Verify FK relationship query works
    joined_sample = pd.read_sql("""
        SELECT b.book_id, b.title, b.price_gbp, b.price_inr, b.rating, c.category_name
        FROM books b
        JOIN categories c ON b.category_id = c.category_id;
    """, connection)
    print("\nNormalized Data Verification:\n", joined_sample)

    connection.close()

Tables in Database:
               name
0       categories
1  sqlite_sequence
2            books

Normalized Data Verification:
    book_id                 title  price_gbp  price_inr  rating  \
0        1  A Light in the Attic      51.77    5461.73       3   
1        2    Tipping the Velvet      53.74    5669.57       1   
2        3            Soumission      50.10    5285.55       1   

        category_name  
0              Poetry  
1  Historical Fiction  
2             Fiction  


5.Using Python's sqlite3 (or pandas.DataFrame.to_sql), insert your cleaned, converted data into this schema. Then write and execute at least 5 SQL queries against the database that collectively demonstrate: SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, and (IN or BETWEEN) — plus at least one JOIN between your two tables (e.g., "list the 10 highest-rated books per category"). Save each query string and its output.



In [ ]:
import sqlite3
import pandas as pd

def create_and_populate_db(db_name: str, cleaned_df: pd.DataFrame) -> sqlite3.Connection:
    """Creates the SQLite database, initializes the schema, and populates data."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    cursor.execute("PRAGMA foreign_keys = ON;")

    # Reset tables for clean execution
    cursor.execute("DROP TABLE IF EXISTS books;")
    cursor.execute("DROP TABLE IF EXISTS categories;")

    # 1. Create Schema
    cursor.execute("""
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT NOT NULL UNIQUE
        );
    """)

    cursor.execute("""
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
            in_stock INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id) REFERENCES categories (category_id)
                ON DELETE CASCADE
        );
    """)

    # 2. Populate Categories Table
    unique_categories = cleaned_df[['category']].drop_duplicates().dropna()
    for _, row in unique_categories.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO categories (category_name) VALUES (?);",
            (row['category'],)
        )
    conn.commit()

    # Map category names to IDs
    cat_df = pd.read_sql("SELECT category_id, category_name FROM categories;", conn)
    cat_map = dict(zip(cat_df['category_name'], cat_df['category_id']))

    # 3. Populate Books Table
    books_tuples = [
        (
            row['title'],
            float(row['price_gbp']),
            float(row['price_inr']),
            int(row['rating']),
            int(row['in_stock']),
            cat_map[row['category']]
        )
        for _, row in cleaned_df.iterrows()
    ]

    cursor.executemany("""
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?);
    """, books_tuples)

    conn.commit()
    return conn


def execute_and_log_queries(conn: sqlite3.Connection, output_file: str = "sql_query_results.txt"):
    """Executes the required SQL queries, prints them, and saves query strings and outputs."""

    # Collection of 5 SQL queries demonstrating all required clauses
    queries = [
        {
            "num": 1,
            "description": "SELECT / WHERE & DISTINCT: List all unique categories available in stock",
            "sql": """
                SELECT DISTINCT c.category_name
                FROM categories c
                JOIN books b ON c.category_id = b.category_id
                WHERE b.in_stock = 1;
            """
        },
        {
            "num": 2,
            "description": "BETWEEN, ORDER BY & LIMIT: Top 5 most affordable books priced between 10 and 25 GBP",
            "sql": """
                SELECT title, price_gbp, price_inr, rating
                FROM books
                WHERE price_gbp BETWEEN 10.0 AND 25.0
                ORDER BY price_gbp ASC
                LIMIT 5;
            """
        },
        {
            "num": 3,
            "description": "IN operator: Select 5-star books belonging to selected genres",
            "sql": """
                SELECT b.title, b.price_gbp, b.rating, c.category_name
                FROM books b
                JOIN categories c ON b.category_id = c.category_id
                WHERE b.rating = 5 AND c.category_name IN ('Poetry', 'Fiction', 'Travel')
                ORDER BY b.price_gbp DESC;
            """
        },
        {
            "num": 4,
            "description": "JOIN, ORDER BY & LIMIT: 10 highest-rated and most expensive books with category names",
            "sql": """
                SELECT b.book_id, b.title, c.category_name, b.rating, b.price_gbp, b.price_inr
                FROM books b
                JOIN categories c ON b.category_id = c.category_id
                ORDER BY b.rating DESC, b.price_gbp DESC
                LIMIT 10;
            """
        },
        {
            "num": 5,
            "description": "JOIN with Aggregation: Average price and count of books per category",
            "sql": """
                SELECT
                    c.category_name,
                    COUNT(b.book_id) AS total_books,
                    ROUND(AVG(b.price_gbp), 2) AS avg_price_gbp,
                    ROUND(AVG(b.price_inr), 2) AS avg_price_inr
                FROM categories c
                JOIN books b ON c.category_id = b.category_id
                GROUP BY c.category_name
                ORDER BY total_books DESC;
            """
        }
    ]

    with open(output_file, "w") as f:
        for q in queries:
            header = f"=== Query {q['num']}: {q['description']} ==="
            sql_clean = q['sql'].strip()

            # Execute query into pandas DataFrame for visual display
            df_result = pd.read_sql(sql_clean, conn)

            # Print to stdout
            print(f"\n{header}\nSQL Query:\n{sql_clean}\n\nOutput:")
            print(df_result.to_string(index=False))
            print("-" * 60)

            # Write to output file
            f.write(f"{header}\n\nSQL Query:\n{sql_clean}\n\nOutput:\n")
            f.write(df_result.to_string(index=False))
            f.write("\n\n" + "="*60 + "\n\n")

    print(f"\nAll queries executed successfully. Results saved to '{output_file}'.")


# --- Pipeline Execution ---
if __name__ == "__main__":
    # Sample cleaned input matching prior module steps
    sample_df = pd.DataFrame({
        'title': [
            'A Light in the Attic', 'Tipping the Velvet', 'Soumission',
            'Sharp Objects', 'Sapiens: A Brief History', 'The Requiem Red',
            'The Dirty Circus', 'The Coming Storm', 'Set Fear on Fire'
        ],
        'price_gbp': [51.77, 53.74, 50.10, 47.82, 54.23, 22.65, 12.11, 14.20, 19.50],
        'price_inr': [5461.73, 5669.57, 5285.55, 5045.01, 5721.27, 2389.58, 1277.61, 1498.10, 2057.25],
        'rating': [3, 1, 1, 4, 5, 1, 5, 2, 5],
        'in_stock': [1, 1, 1, 1, 1, 1, 0, 1, 1],
        'category': ['Poetry', 'Historical Fiction', 'Fiction', 'Mystery', 'History', 'Young Adult', 'Poetry', 'Travel', 'Travel']
    })

    # Initialize DB and run tasks
    db_connection = create_and_populate_db("zepto_books.db", sample_df)
    execute_and_log_queries(db_connection, "sql_query_results.txt")
    db_connection.close()


=== Query 1: SELECT / WHERE & DISTINCT: List all unique categories available in stock ===
SQL Query:
SELECT DISTINCT c.category_name
                FROM categories c
                JOIN books b ON c.category_id = b.category_id
                WHERE b.in_stock = 1;

Output:
     category_name
            Poetry
Historical Fiction
           Fiction
           Mystery
           History
       Young Adult
            Travel
------------------------------------------------------------

=== Query 2: BETWEEN, ORDER BY & LIMIT: Top 5 most affordable books priced between 10 and 25 GBP ===
SQL Query:
SELECT title, price_gbp, price_inr, rating
                FROM books
                WHERE price_gbp BETWEEN 10.0 AND 25.0
                ORDER BY price_gbp ASC
                LIMIT 5;

Output:
           title  price_gbp  price_inr  rating
The Dirty Circus      12.11    1277.61       5
The Coming Storm      14.20    1498.10       2
Set Fear on Fire      19.50    2057.25       5
 The Requiem

6.Read back at least two of the above query results into pandas DataFrames using pd.read_sql(...), and separately reproduce the join-query's result using pd.merge(...) directly on your in-memory DataFrames (no SQL) — show that both approaches produce equivalent output.

In [ ]:
import os
import seaborn as sns
import pandas as pd

# Ensure the /analytics directory exists
os.makedirs("analytics", exist_ok=True)
csv_fallback_path = os.path.join("analytics", "titanic.csv")

# 1. Load the raw dataset (Single entry point from Seaborn or committed CSV fallback)
try:
    df = sns.load_dataset('titanic')
    print("Successfully loaded 'titanic' dataset via Seaborn.")
except Exception as e:
    print(f"Network/Load error ({e}). Loading from local fallback CSV...")
    df = pd.read_csv(csv_fallback_path)

# 2. Save the loaded DataFrame immediately as the required offline fallback
df.to_csv(csv_fallback_path, index=False)
print(f"Saved offline fallback dataset to '{csv_fallback_path}'.\n")

# 3. Profile the Dataset
print("=" * 50)
print("1. DATASET SHAPE")
print("=" * 50)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n" + "=" * 50)
print("2. DATASET INFO")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("3. DESCRIPTIVE STATISTICS")
print("=" * 50)
print(df.describe(include='all'))

# 4. Compute and report missing value percentages
print("\n" + "=" * 50)
print("4. MISSING VALUES REPORT")
print("=" * 50)

missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Percentage (%)': missing_percentages.round(2)
})

# Filter and display only columns with missing values (> 0%)
missing_columns_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Percentage (%)', ascending=False)

if not missing_columns_df.empty:
    print(missing_columns_df)
else:
    print("No missing values found in any column.")

Successfully loaded 'titanic' dataset via Seaborn.
Saved offline fallback dataset to 'analytics/titanic.csv'.

1. DATASET SHAPE
Rows: 891, Columns: 15

2. DATASET INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool 

In [ ]:
import sqlite3
import pandas as pd

# 1. Connect to your SQLite database
conn = sqlite3.connect("zepto_books.db")

# -------------------------------------------------------------------------
# Step A: Read back raw tables into DataFrames using pd.read_sql
# -------------------------------------------------------------------------
# Table 1: Categories
df_categories = pd.read_sql("SELECT * FROM categories;", conn)

# Table 2: Books
df_books = pd.read_sql("SELECT * FROM books;", conn)

# -------------------------------------------------------------------------
# Step B: Read back the SQL JOIN query directly via pd.read_sql
# -------------------------------------------------------------------------
sql_join_query = """
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id;
"""

df_sql_join = pd.read_sql(sql_join_query, conn)

# Close database connection
conn.close()

# -------------------------------------------------------------------------
# Step C: Reproduce the JOIN using pd.merge directly in pandas (No SQL)
# -------------------------------------------------------------------------
df_pandas_merge = pd.merge(
    df_books,
    df_categories,
    on="category_id",
    how="inner"
)[["book_id", "title", "price_gbp", "price_inr", "rating", "in_stock", "category_name"]]

# -------------------------------------------------------------------------
# Step D: Verify equivalence between SQL JOIN and pandas pd.merge
# -------------------------------------------------------------------------
# Sort both DataFrames by book_id to ensure consistent row ordering
df_sql_sorted = df_sql_join.sort_values(by="book_id").reset_index(drop=True)
df_merge_sorted = df_pandas_merge.sort_values(by="book_id").reset_index(drop=True)

# Check exact equivalence
are_equal = df_sql_sorted.equals(df_merge_sorted)
print(f"Are both outputs identical? {are_equal}")

# Optional: Assert equality for automated testing/scripts
pd.testing.assert_frame_equal(df_sql_sorted, df_merge_sorted)
print("Assertion Passed: Both SQL JOIN and pd.merge produce equivalent DataFrames!")

Are both outputs identical? True
Assertion Passed: Both SQL JOIN and pd.merge produce equivalent DataFrames!


In [ ]:
from IPython.display import display

print("--- SQL JOIN Output (First 5 Rows) ---")
display(df_sql_sorted.head())

print("--- pandas pd.merge Output (First 5 Rows) ---")
display(df_merge_sorted.head())

--- SQL JOIN Output (First 5 Rows) ---


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,A Light in the Attic,51.77,5461.73,3,1,Poetry
1,2,Tipping the Velvet,53.74,5669.57,1,1,Historical Fiction
2,3,Soumission,50.10,5285.55,1,1,Fiction
3,4,Sharp Objects,47.82,5045.01,4,1,Mystery
4,5,Sapiens: A Brief History,54.23,5721.27,5,1,History


--- pandas pd.merge Output (First 5 Rows) ---


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,A Light in the Attic,51.77,5461.73,3,1,Poetry
1,2,Tipping the Velvet,53.74,5669.57,1,1,Historical Fiction
2,3,Soumission,50.10,5285.55,1,1,Fiction
3,4,Sharp Objects,47.82,5045.01,4,1,Mystery
4,5,Sapiens: A Brief History,54.23,5721.27,5,1,History
